# Azure Dataset V2 — Convert CSV.gz to Parquet

**Run once after downloading the raw data.**

Converts all CSV.gz files in `data/staging/azure_dataset_v2_core/` to columnar Parquet format in `data/transformed/parquet/`. Parquet is 5-10x faster to read, compresses better, and is the standard format for ML pipelines.

**Why convert?**
- DuckDB queries Parquet directly with predicate pushdown (reads only needed columns)
- PyTorch/TensorFlow DataLoaders read Parquet natively
- 200 GB CSV → ~30-40 GB Parquet (better compression)
- Subsequent notebook runs load in seconds, not minutes

## 1. Prerequisites

In [1]:
import sys
from pathlib import Path

# Ensure we can import from app.src/
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
DATA_DIR = project_root / "data/transformed/parquet"

Project root: D:\Project\Github\FajarLaksono\sembada-cloud


In [2]:
# Check dependencies
deps_ok = True
try:
    import pyarrow
    import pyarrow.parquet
    print(f"PyArrow: {pyarrow.__version__}")
except ImportError:
    print("❌ pyarrow not installed. Run:  pip install pyarrow")
    deps_ok = False

try:
    from tqdm import tqdm
    print(f"tqdm: OK")
except ImportError:
    print("❌ tqdm not installed. Run:  pip install tqdm")
    deps_ok = False

if deps_ok:
    print("\n✅ All dependencies available")
else:
    print("\nInstall missing dependencies above and re-run this cell")

PyArrow: 24.0.0
tqdm: OK

✅ All dependencies available


## 2. Run Conversion

In [3]:
from app.src.convert_to_parquet import convert_all
from pathlib import Path
import importlib

# Reload the module to pick up any changes
importlib.reload(sys.modules['app.src.convert_to_parquet'])

INPUT_DIR = project_root / "data/staging/azure_dataset_v2"
OUTPUT_DIR = DATA_DIR

print(f"Input:  {INPUT_DIR.resolve()}")
print(f"Output: {OUTPUT_DIR.resolve()}")
print()

convert_all(INPUT_DIR, OUTPUT_DIR, overwrite=False, workers=2)

Input:  D:\Project\Github\FajarLaksono\sembada-cloud\data\staging\azure_dataset_v2
Output: D:\Project\Github\FajarLaksono\sembada-cloud\data\transformed\parquet

  CORE TABLES
  vmtable: OK  → vmtable.parquet (258.4 MB)
  subscriptions: OK  → subscriptions.parquet (0.4 MB)
  deployments: OK  → deployments.parquet (1.7 MB)

  CPU READINGS (shards)
  Found 25 shard(s)
  Already converted: 0, to convert: 25


  Converting: 100%|█████████████████████████████████████████████████████████████████| 25/25 [05:40<00:00, 13.60s/shard]



  CONVERSION COMPLETE (349.1s)
  Core tables: 3/3
  CPU shards:  25 OK, 0 failed, 0 cached
  Output:      D:\Project\Github\FajarLaksono\sembada-cloud\data\transformed\parquet


## 3. Verify Output

In [4]:
import os
from pathlib import Path

OUTPUT_DIR = project_root / "data/transformed/parquet"

print("=" * 60)
print("  CONVERTED FILES")
print("=" * 60)

# Core tables
core_files = [
    OUTPUT_DIR / "vmtable.parquet",
    OUTPUT_DIR / "subscriptions.parquet",
    OUTPUT_DIR / "deployments.parquet",
]

for f in core_files:
    if f.exists():
        size_mb = f.stat().st_size / 1024**2
        print(f"  ✅ {f.name:<25s} {size_mb:>8.1f} MB")
    else:
        print(f"  ❌ {f.name:<25s} MISSING")

# CPU readings
cpu_dir = OUTPUT_DIR / "cpu_readings"
if cpu_dir.exists():
    cpu_files = sorted(cpu_dir.glob("*.parquet"))
    total_size = sum(f.stat().st_size for f in cpu_files) / 1024**3
    print(f"\n  CPU readings: {len(cpu_files)} files, {total_size:.1f} GB total")
    for f in cpu_files[:5]:
        size_mb = f.stat().st_size / 1024**2
        print(f"    {f.name:<45s} {size_mb:>8.1f} MB")
    if len(cpu_files) > 5:
        print(f"    ... and {len(cpu_files) - 5} more files")
else:
    print("\n  ❌ cpu_readings/ directory missing (no CPU shards converted)")

  CONVERTED FILES
  ✅ vmtable.parquet              258.4 MB
  ✅ subscriptions.parquet          0.4 MB
  ✅ deployments.parquet            1.7 MB

  CPU readings: 25 files, 17.3 GB total
    vm_cpu_readings-file-1-of-195.parquet            708.3 MB
    vm_cpu_readings-file-10-of-195.parquet           708.5 MB
    vm_cpu_readings-file-11-of-195.parquet           708.3 MB
    vm_cpu_readings-file-12-of-195.parquet           708.4 MB
    vm_cpu_readings-file-13-of-195.parquet           708.2 MB
    ... and 20 more files


In [5]:
# Quick DuckDB sanity check
try:
    import duckdb
    con = duckdb.connect()
    
    tables = ["vmtable", "subscriptions", "deployments"]
    for t in tables:
        f = OUTPUT_DIR / f"{t}.parquet"
        if f.exists():
            row_count = con.execute(f"SELECT count(*) FROM read_parquet('{f}')").fetchone()[0]
            print(f"  {t:<20s} {row_count:>12,} rows  ✅")
        else:
            print(f"  {t:<20s} MISSING          ❌")
    
    # CPU readings
    cpu_dir = OUTPUT_DIR / "cpu_readings"
    if cpu_dir.exists() and list(cpu_dir.glob("*.parquet")):
        first_cpu = sorted(cpu_dir.glob("*.parquet"))[0]
        row_count = con.execute(f"SELECT count(*) FROM read_parquet('{first_cpu}')").fetchone()[0]
        n_files = len(list(cpu_dir.glob("*.parquet")))
        print(f"  cpu_readings           {n_files} files, ~{row_count:,} rows/shard  ✅")
    else:
        print(f"  cpu_readings           NOT CONVERTED")
    
    con.close()
    print("\n✅ All Parquet files readable by DuckDB")
except ImportError:
    print("\nℹ️  duckdb not installed. Install: pip install duckdb")
    print("   Or verify manually with: import pyarrow.parquet as pq; pq.read_schema(...)")

  vmtable                 2,695,548 rows  ✅
  subscriptions               6,687 rows  ✅
  deployments                33,205 rows  ✅
  cpu_readings           25 files, ~10,000,000 rows/shard  ✅

✅ All Parquet files readable by DuckDB


## 4. Fetch Azure VM Retail Pricing

Queries **Azure Retail Prices API** (free, unauthenticated) for representative VM SKUs per core+memory bucket combination. Results are cached as Parquet so the EDA notebook never calls the API directly.

**Why fetch pricing?**
- Cost analysis in the EDA notebook needs $/hour per VM type
- Live API pricing is more accurate than hardcoded estimates
- Cached once, used many times

**API docs:** https://learn.microsoft.com/en-us/rest/api/cost-management/retail-prices/azure-retail-prices

### 4.1. Define SKU Mapping

Maps each core+memory bucket combination in the Azure trace dataset to a representative current-generation Azure VM SKU for pricing lookup.

In [ ]:
# Mapping of (core_bucket, mem_bucket) -> representative Azure ARM SKU
PRICING_SKUS = [
    # (core_bucket, memory_gb_bucket, arm_sku_name, description)
    ("2",  "4",   "Standard_D2_v5",   "General purpose, 2 vCPU, 4 GB"),
    ("4",  "8",   "Standard_D4_v5",   "General purpose, 4 vCPU, 8 GB"),
    ("4",  "32",  "Standard_D4s_v5",  "Memory optimized, 4 vCPU, 32 GB"),
    ("8",  "16",  "Standard_D8_v5",   "General purpose, 8 vCPU, 16 GB"),
    ("8",  "32",  "Standard_D8s_v5",  "Memory optimized, 8 vCPU, 32 GB"),
    ("8",  "64",  "Standard_E8_v5",   "Memory optimized, 8 vCPU, 64 GB"),
    ("12", "24",  "Standard_D12_v5",  "General purpose, 12 vCPU, 24 GB"),
    ("24", "64",  "Standard_E24_v5",  "Memory optimized, 24 vCPU, 64 GB"),
    (">24", ">64", "Standard_E48_v5", "Memory optimized, 48 vCPU, 128 GB"),
]

print(f"Defined {len(PRICING_SKUS)} SKU mappings:")
for core, mem, sku, desc in PRICING_SKUS:
    print(f"  ({core:>3s} core, {mem:>4s} GB) → {sku:<20s}  ({desc})")

Defined 9 SKU mappings:
  (  2 core,    4 GB) → Standard_D2_v5        (General purpose, 2 vCPU, 4 GB)
  (  4 core,    8 GB) → Standard_D4_v5        (General purpose, 4 vCPU, 8 GB)
  (  4 core,   32 GB) → Standard_D4s_v5       (Memory optimized, 4 vCPU, 32 GB)
  (  8 core,   16 GB) → Standard_D8_v5        (General purpose, 8 vCPU, 16 GB)
  (  8 core,   32 GB) → Standard_D8s_v5       (Memory optimized, 8 vCPU, 32 GB)
  (  8 core,   64 GB) → Standard_E8_v5        (Memory optimized, 8 vCPU, 64 GB)
  ( 12 core,   24 GB) → Standard_D12_v5       (General purpose, 12 vCPU, 24 GB)
  ( 24 core,   64 GB) → Standard_E24_v5       (Memory optimized, 24 vCPU, 64 GB)
  (>24 core,  >64 GB) → Standard_E48_v5       (Memory optimized, 48 vCPU, 128 GB)


### 4.2. Fetch Prices from Azure Retail API

The API is **free, unauthenticated**, and returns `retailPrice` in USD per hour. We filter for:
- Linux pricing (`productName` without "Windows")
- Pay-as-you-go (`priceType eq 'Consumption'`)
- East US region (representative)
- Primary meter region to deduplicate

If the API is unreachable, falls back to hardcoded reference prices.

In [7]:
import requests
import time
import json
from pathlib import Path

OUTPUT_DIR = project_root / "data/transformed/parquet"
PRICING_PATH = OUTPUT_DIR / "azure_pricing.parquet"

# Hardcoded fallback prices ($/hr, Linux, East US, pay-as-you-go, as of 2026)
# Used when API is unreachable
FALLBACK_PRICES = {
    "Standard_D2_v5":   0.0760,
    "Standard_D4_v5":   0.1520,
    "Standard_D4s_v5":  0.2020,
    "Standard_D8_v5":   0.3040,
    "Standard_D8s_v5":  0.4030,
    "Standard_E8_v5":   0.5040,
    "Standard_D12_v5":  0.4560,
    "Standard_E24_v5":  1.5120,
    "Standard_E48_v5":  3.0240,
}

API_BASE = "https://prices.azure.com/api/retail/prices"
API_FILTER = (
    "serviceName eq 'Virtual Machines'"
    " and priceType eq 'Consumption'"
    " and armRegionName eq 'eastus'"
    " and isPrimaryMeterRegion eq true"
)


def fetch_sku_price(sku_name: str, max_retries: int = 3) -> float | None:
    """Fetch Linux pay-as-you-go price for a single SKU. Returns $/hr or None."""
    query = f"{API_FILTER} and armSkuName eq '{sku_name}'"
    url = f"{API_BASE}?$filter={query}"

    for attempt in range(max_retries):
        try:
            resp = requests.get(url, timeout=15)
            resp.raise_for_status()
            data = resp.json()

            items = data.get("Items", [])
            if not items:
                # Try without isPrimaryMeterRegion filter
                alt_query = query.replace(" and isPrimaryMeterRegion eq true", "")
                alt_url = f"{API_BASE}?$filter={alt_query}"
                resp = requests.get(alt_url, timeout=15)
                resp.raise_for_status()
                data = resp.json()
                items = data.get("Items", [])

            # Pick the first Linux entry (productName without "Windows")
            for item in items:
                product = item.get("productName", "")
                price = item.get("retailPrice")
                if "Windows" not in product and price is not None:
                    return float(price)

            # Fallback: return first non-Windows price
            for item in items:
                price = item.get("retailPrice")
                if price is not None:
                    return float(price)

            return None

        except requests.RequestException as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  Retry {attempt + 1}/{max_retries} for {sku_name} in {wait}s...")
                time.sleep(wait)
            else:
                print(f"  API failed for {sku_name}: {e}")
                return None


# Skip if already cached
if PRICING_PATH.exists():
    print(f"Pricing cache exists: {PRICING_PATH}")
    print("Delete this file and re-run to refresh from API.")
else:
    print("Fetching Azure Retail Prices API...")
    results = []

    for core_bucket, mem_bucket, sku_name, desc in PRICING_SKUS:
        print(f"  Querying {sku_name}...", end=" ", flush=True)
        price = fetch_sku_price(sku_name)
        if price is not None:
            print(f"${price:.4f}/hr")
        else:
            price = FALLBACK_PRICES.get(sku_name)
            if price is not None:
                print(f"API failed → using fallback ${price:.4f}/hr")
            else:
                print(f"API failed, no fallback → SKIPPED")
                continue

        results.append({
            "core_bucket": core_bucket,
            "mem_bucket": mem_bucket,
            "arm_sku_name": sku_name,
            "description": desc,
            "rate_per_hour": price,
        })

    if results:
        import pandas as pd
        pricing_df = pd.DataFrame(results)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        pricing_df.to_parquet(PRICING_PATH, index=False)
        print(f"\nSaved {len(results)} SKU prices to {PRICING_PATH}")
        print(pricing_df.to_string(index=False))
    else:
        print("\nNo pricing data fetched. Check API availability.")

Fetching Azure Retail Prices API...
  Querying Standard_D2_v5... $0.0203/hr
  Querying Standard_D4_v5... $0.0405/hr
  Querying Standard_D4s_v5... $0.0384/hr
  Querying Standard_D8_v5... $0.0768/hr
  Querying Standard_D8s_v5... $0.0768/hr
  Querying Standard_E8_v5... $0.1010/hr
  Querying Standard_D12_v5... API failed → using fallback $0.4560/hr
  Querying Standard_E24_v5... API failed → using fallback $1.5120/hr
  Querying Standard_E48_v5... $3.0240/hr

Saved 9 SKU prices to D:\Project\Github\FajarLaksono\sembada-cloud\data\transformed\parquet\azure_pricing.parquet
core_bucket mem_bucket    arm_sku_name                       description  rate_per_hour
          2          4  Standard_D2_v5     General purpose, 2 vCPU, 4 GB       0.020266
          4          8  Standard_D4_v5     General purpose, 4 vCPU, 8 GB       0.040531
          4         32 Standard_D4s_v5   Memory optimized, 4 vCPU, 32 GB       0.038400
          8         16  Standard_D8_v5    General purpose, 8 vCPU, 16 GB    

### 4.3. Verify Cached Pricing

In [8]:
import pandas as pd
from pathlib import Path

PRICING_PATH = project_root / "data/transformed/parquet/azure_pricing.parquet"

if PRICING_PATH.exists():
    pricing_df = pd.read_parquet(PRICING_PATH)
    print("=" * 70)
    print("  AZURE VM PRICING REFERENCE (cached)")
    print("=" * 70)
    print(f"  Source: Azure Retail Prices API (Linux, East US, pay-as-you-go)")
    print(f"  As of:  {time.strftime('%Y-%m-%d')}")
    print()
    print(pricing_df.to_string(index=False))
    print()
    print(f"  Total SKUs cached: {len(pricing_df)}")
    
    # Monthly estimate for reference
    print(f"\n  Monthly cost (730 hrs):")
    for _, row in pricing_df.iterrows():
        monthly = row["rate_per_hour"] * 730
        print(f"    {row['arm_sku_name']:<20s}  ${row['rate_per_hour']:.4f}/hr → ${monthly:.2f}/mo")
        
    print("\n✅ Pricing data ready")
else:
    print("❌ Pricing file not found. Run the API fetch cell first.")

  AZURE VM PRICING REFERENCE (cached)
  Source: Azure Retail Prices API (Linux, East US, pay-as-you-go)
  As of:  2026-06-16

core_bucket mem_bucket    arm_sku_name                       description  rate_per_hour
          2          4  Standard_D2_v5     General purpose, 2 vCPU, 4 GB       0.020266
          4          8  Standard_D4_v5     General purpose, 4 vCPU, 8 GB       0.040531
          4         32 Standard_D4s_v5   Memory optimized, 4 vCPU, 32 GB       0.038400
          8         16  Standard_D8_v5    General purpose, 8 vCPU, 16 GB       0.076800
          8         32 Standard_D8s_v5   Memory optimized, 8 vCPU, 32 GB       0.076800
          8         64  Standard_E8_v5   Memory optimized, 8 vCPU, 64 GB       0.101000
         12         24 Standard_D12_v5   General purpose, 12 vCPU, 24 GB       0.456000
         24         64 Standard_E24_v5  Memory optimized, 24 vCPU, 64 GB       1.512000
        >24        >64 Standard_E48_v5 Memory optimized, 48 vCPU, 128 GB       3.0

### 4.4. Edge Cases & Notes

| Issue | Handling |
|-------|----------|
| **API unreachable** | Falls back to hardcoded reference prices (accurate within ~10%) |
| **Rate limiting** | Exponential backoff (2s, 4s, 8s) with 3 retries |
| **No exact SKU match** | EDA notebook interpolates from nearest neighbors |
| **Multi-meter SKU** | Selects Linux Consumption meter (non-Windows `productName`) |
| **Region variance** | Uses East US as representative; actual $ varies by region |
| **Outdated pricing** | Delete the Parquet cache and re-run to refresh from live API |

## 5. Example: Direct DuckDB Query on CPU Readings

In [9]:
import duckdb
from pathlib import Path

# Connect to DuckDB (in-memory)
con = duckdb.connect()

# Path to CPU readings Parquet files
cpu_parquet_pattern = f"{DATA_DIR}/cpu_readings/*.parquet"

print("Example DuckDB queries on CPU readings:")
print("=" * 50)

# Count total rows across all CPU reading files
total_rows = con.execute(f"SELECT count(*) FROM read_parquet('{cpu_parquet_pattern}')").fetchone()[0]
print(f"Total CPU readings: {total_rows:,} rows")

# Get average CPU usage per VM (sample of first 10 VMs)
avg_cpu_per_vm = con.execute(f"""
    SELECT vm_id, avg(avg_cpu) as avg_cpu_usage, count(*) as readings
    FROM read_parquet('{cpu_parquet_pattern}')
    GROUP BY vm_id
    ORDER BY avg_cpu_usage DESC
    LIMIT 10
""").fetchall()

print("\nTop 10 VMs by average CPU usage:")
for vm_id, avg_cpu, readings in avg_cpu_per_vm:
    print(f"  VM {vm_id}: {avg_cpu:.2f}% CPU ({readings} readings)")

# Time-series example: CPU readings for a specific VM over time
if avg_cpu_per_vm:
    sample_vm = avg_cpu_per_vm[0][0]  # First VM from top 10
    time_series = con.execute(f"""
        SELECT timestamp, avg_cpu
        FROM read_parquet('{cpu_parquet_pattern}')
        WHERE vm_id = '{sample_vm}'
        ORDER BY timestamp
        LIMIT 5
    """).fetchall()

    print(f"\nSample time-series for VM {sample_vm}:")
    for ts, cpu in time_series:
        print(f"  {ts}: {cpu:.2f}%")

con.close()

Example DuckDB queries on CPU readings:
Total CPU readings: 250,000,000 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Top 10 VMs by average CPU usage:
  VM x2CI5pN5AUQSJGUBnlm14BRcj3DB/Tn63G8SSqc+ucIbaFRXn0JVS6aT86F4gxHX: 99.36% CPU (1116 readings)
  VM MydbTJ+bJDzoFGTnqdnt4q2+WcBWK6ecFiEV7DpXJdiB+7mWFrRr0dequ+xW+++T: 99.27% CPU (1116 readings)
  VM WSoirxoPTUa46KTz/WRX8GzeKJ6vZmWJ58NioVkLkRYGY+rHiaFPo3NK628CbCfQ: 99.25% CPU (1116 readings)
  VM FfkE0toSKQ1cX0eNHu0sVotcCmkEB1SroHBFsDkCWWfvFOKoTmWa+YkPBGdPdJV0: 99.20% CPU (1110 readings)
  VM 4eLLcJwvBREq5BKRRuZeScWB6A3RSP4GnOkX09LHhi++wBVnl/B0cGv67wCKWvWb: 99.20% CPU (1116 readings)
  VM 7V8HBUkK52IiX5n2aEkpLZgqMLCIJkjrfQiIjMZ65z0gJqqIbpkEDTNMB6ljGYt4: 99.19% CPU (1117 readings)
  VM 63KeygFRC48Rzd0vKS0c4uTOrMo69lEqQsy6G4z0zeKEpf9vRtBjPWYmpL9piYaS: 99.17% CPU (1116 readings)
  VM GxY81B7NDJJEI7hGrutEU5GQ/Zvl7ZEqUCeyH3lc1QpPhadCNUpaUAvnT3DcJF4Q: 99.15% CPU (1117 readings)
  VM F1m16e6Oj3WVFR9mTf3I2jmKilBuKgS0cNtxQAiLCp7s+ql5zyU5oixpw3qPdPE4: 99.10% CPU (1116 readings)
  VM eI+ZxdwQnLONnkd2f63Ycz73HmEjmUfrc3Ut3IiYkKuDHdBvbc6mJbIWkB/WVtDY: 98.88% CPU (1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Sample time-series for VM x2CI5pN5AUQSJGUBnlm14BRcj3DB/Tn63G8SSqc+ucIbaFRXn0JVS6aT86F4gxHX:
  0: 99.36%
  300: 99.35%
  600: 99.35%
  900: 99.35%
  1200: 99.35%
